# T12 vertebral-body midpoint audit (Colab)

**Notebook v1.2.0** | 2026-09-12 | PSS-aware high-frequency telemetry and selective minimal-mask archiving

Validated for T4 High-RAM. Each CT is staged just in time on Colab local disk. Per-case results are atomically checkpointed to Drive. A 5-second local sampler records RSS/PSS/USS, process count, active case/task, GPU and disk metrics, with bounded-loss atomic mirroring to Drive. Console output is intentionally concise.

`SUCCESS` and deterministic `QC_ERROR` rows are terminal on resume; transient `PROCESS_ERROR` rows are retried. The default observed rate is 1.27 CU/h and can be edited in Cell 2.

## Changelog

| Version | Date | Change |
|---|---|---|
| v1.0.0 | 2026-09-11 | Initial checkpointed Colab launcher |
| v1.0.1 | 2026-09-11 | Colab Secret license activation |
| v1.0.2 | 2026-09-11 | Pin TotalSegmentator 2.18.0 and probe tasks |
| v1.1.0 | 2026-09-11 | Telemetry, concise logs, terminal QC status, retryable process status, local scratch |
| v1.1.1 | 2026-09-11 | Close the Drive telemetry file after every sample so abrupt runtime loss preserves recent records |
| v1.1.2 | 2026-09-11 | Read the runner from `assets/run_t12_audit.py` on Drive, falling back to GitHub raw only when it is absent. v1.1.1 fetched it from an unauthenticated raw.githubusercontent URL, which would 404 the moment this repository became private. The source and its sha256 are now printed. |
| v1.2.0 | 2026-09-12 | Add 5-second RSS/PSS/USS telemetry linked to session/case/task state, atomic 30-second Drive mirrors, stage/measurement/cleanup timings and opt-in minimal T11/T12/L1 mask archiving. Existing checkpoints remain compatible. |

> Reopen with `?flush_cache=true` after an update. An already-running older Cell 6 may finish; its Drive checkpoint is compatible with resume.


In [ ]:
# Cell 1: environment, license, Drive and assets
import csv, hashlib, importlib.metadata, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'TotalSegmentator==2.18.0', 'SimpleITK', 'scipy', 'psutil'], check=True)
from google.colab import userdata, drive
license_number = None
for name in ('TOTALSEG_LICENSE', 'totalseg_license', 'TOTALSEGMENTATOR_LICENSE'):
    try: license_number = userdata.get(name)
    except Exception: pass
    if license_number: break
assert license_number, 'BLOCKING: enable TOTALSEG_LICENSE for this notebook'
subprocess.run([shutil.which('totalseg_set_license'), '-l', license_number], check=True, capture_output=True, text=True)
del license_number
NOTEBOOK_VERSION = '1.2.0'; SESSION_ID = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + f'-{os.getpid()}'; print('Notebook', NOTEBOOK_VERSION, 'session', SESSION_ID)
drive.mount('/content/drive', force_remount=False)
GDRIVE = Path('/content/drive/MyDrive/cardiac_colab')
TASK_DIR = GDRIVE/'t12_midpoint_audit_20260911'; ASSET_DIR = TASK_DIR/'assets'; OUTPUT_DIR = TASK_DIR/'output'; LOG_DIR = TASK_DIR/'logs'
for p in (OUTPUT_DIR, LOG_DIR): p.mkdir(parents=True, exist_ok=True)
MANIFEST = ASSET_DIR/'full_manifest_colab_20260911.csv'; SELECTIVE_MANIFEST = ASSET_DIR/'selective_rerun_manifest_20260912.csv'; LABEL_ARCHIVE = ASSET_DIR/'t12_existing_labels_20260911.tar.gz'
CHECKPOINT_FILE = OUTPUT_DIR/'results.csv'; EVENTS_FILE = LOG_DIR/'case_task_events.jsonl'; TELEMETRY_FILE = LOG_DIR/f'resource_telemetry_{SESSION_ID}.csv'
RUNNER = Path('/content/run_t12_audit.py'); RUNNER_ASSET = ASSET_DIR/'run_t12_audit.py'
RUNNER_URL = 'https://raw.githubusercontent.com/zhurong2020/claude-colab-projects/main/standalone/female-early-chd-t12/run_t12_audit.py'
if RUNNER_ASSET.is_file():
    shutil.copyfile(RUNNER_ASSET, RUNNER); RUNNER_SOURCE = f'Drive {RUNNER_ASSET}'
else:
    urllib.request.urlretrieve(RUNNER_URL, RUNNER); RUNNER_SOURCE = 'GitHub raw (requires a public repository)'
RUNNER.chmod(0o755)
print('Runner from', RUNNER_SOURCE, '| sha256', hashlib.sha256(RUNNER.read_bytes()).hexdigest()[:16])
SCRATCH_ROOT = Path('/mnt/local-scratch') if Path('/mnt/local-scratch').is_dir() else Path('/content')
WORK_DIR = SCRATCH_ROOT/'t12_midpoint_audit'; LABEL_DIR = Path('/content/t12_audit_labels'); ACTIVE_STATE = WORK_DIR/'active_state.json'; LOCAL_TELEMETRY = WORK_DIR/f'resource_telemetry_{SESSION_ID}.csv'; WORK_DIR.mkdir(parents=True, exist_ok=True)
assert MANIFEST.is_file() and LABEL_ARCHIVE.is_file() and RUNNER.is_file(), 'BLOCKING: missing assets'
if not LABEL_DIR.is_dir():
    with tarfile.open(LABEL_ARCHIVE, 'r:gz') as tf: tf.extractall('/content')
rows = list(csv.DictReader(MANIFEST.open()))
missing = [str(p) for r in rows for p in (Path(r['image_path']), Path(r['label_dir'])/'vertebrae_T12.nii.gz', Path(r['label_dir'])/'tissue_4types_torso_fat.nii.gz') if not p.is_file()]
print(f'Manifest={len(rows)} missing={len(missing)} scratch={SCRATCH_ROOT}'); assert not missing, missing[:20]


In [ ]:
# Cell 2: resource gate and session provenance
import psutil, torch
CU_PER_HOUR = 1.27  # edit if Colab displays a different live rate
RUN_SELECTIVE_QC = False  # set True only for the frozen post-run QC manifest
assert torch.cuda.is_available(), 'BLOCKING: select GPU runtime'
gpu_name = torch.cuda.get_device_name(0); vram_gb = torch.cuda.get_device_properties(0).total_memory/2**30
ram_gb = psutil.virtual_memory().total/2**30; disk = shutil.disk_usage(SCRATCH_ROOT)
print(f'GPU={gpu_name}; VRAM={vram_gb:.1f} GiB; RAM={ram_gb:.1f} GiB; scratch_free={disk.free/2**30:.1f} GiB; rate={CU_PER_HOUR} CU/h')
assert ram_gb >= 20 and disk.free/2**30 >= 20, 'BLOCKING: High-RAM/local scratch requirement not met'
session = {'session_id': SESSION_ID, 'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'notebook_version': NOTEBOOK_VERSION, 'gpu': gpu_name, 'vram_gib': vram_gb, 'ram_gib': ram_gb, 'scratch_total_gib': disk.total/2**30, 'cu_per_hour': CU_PER_HOUR, 'totalsegmentator': importlib.metadata.version('TotalSegmentator'), 'python': sys.version, 'telemetry_interval_seconds': 5, 'telemetry_drive_mirror_seconds': 30, 'memory_metric_note': 'PSS/USS are sampled estimates; Colab telemetry is not a cgroup kernel gold standard'}
(LOG_DIR/f'session_provenance_{SESSION_ID}.json').write_text(json.dumps(session, indent=2))


In [ ]:
# Cell 3: task probe, concise output and PSS-aware bounded-loss telemetry
import threading
tasks = subprocess.run(['TotalSegmentator', '--list-tasks'], text=True, capture_output=True, check=True).stdout
assert all(x in tasks for x in ('vertebrae_body', 'vertebrae_pp')), 'BLOCKING: required task unavailable'
def gpu_sample():
    q = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw', '--format=csv,noheader,nounits'], text=True, capture_output=True)
    try: return [float(x.strip()) for x in q.stdout.strip().split(',')]
    except Exception: return [None]*5
def mirror_telemetry():
    if not LOCAL_TELEMETRY.exists(): return
    tmp=TELEMETRY_FILE.with_suffix('.csv.tmp'); shutil.copyfile(LOCAL_TELEMETRY,tmp); os.replace(tmp,TELEMETRY_FILE)
def monitor(pid, stop, checkpoint_file):
    fields=['session_id','sample_seq','timestamp_utc','elapsed_monotonic_s','runner_pid','process_count','root_rss_gib','tree_rss_gib','tree_pss_gib','tree_uss_gib','pss_processes','active_patientingroupid','active_case_id','active_task','active_phase','system_used_gib','system_available_gib','system_percent','gpu_util_percent','gpu_mem_used_mib','gpu_mem_total_mib','gpu_temp_c','gpu_power_w','scratch_used_gib','scratch_free_gib','checkpoint_rows']
    started=time.monotonic(); last_mirror=0; last_state=''; seq=0; LOCAL_TELEMETRY.unlink(missing_ok=True)
    while not stop.is_set():
        seq+=1; rss=pss=uss=root_rss=0; pss_n=0; processes=[]
        try:
            root=psutil.Process(pid); processes=[root]+root.children(recursive=True); root_rss=root.memory_info().rss
            for p in processes:
                if not p.is_running(): continue
                rss+=p.memory_info().rss
                try: full=p.memory_full_info(); pss+=full.pss; uss+=full.uss; pss_n+=1
                except (psutil.AccessDenied,psutil.NoSuchProcess,AttributeError): pass
        except (psutil.NoSuchProcess,psutil.AccessDenied): pass
        try: state=json.loads(ACTIVE_STATE.read_text())
        except Exception: state={}
        state_key='|'.join(str(state.get(k,'')) for k in ('case_id','task','phase'))
        vm=psutil.virtual_memory(); ds=shutil.disk_usage(SCRATCH_ROOT); gu,gm,gt,temp,power=gpu_sample()
        try: nrows=max(0,sum(1 for _ in checkpoint_file.open())-1)
        except Exception: nrows=0
        values=[SESSION_ID,seq,time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),round(time.monotonic()-started,3),pid,len(processes),root_rss/2**30,rss/2**30,(pss/2**30 if pss_n else ''),(uss/2**30 if pss_n else ''),pss_n,state.get('patientingroupid',''),state.get('case_id',''),state.get('task',''),state.get('phase',''),vm.used/2**30,vm.available/2**30,vm.percent,gu,gm,gt,temp,power,ds.used/2**30,ds.free/2**30,nrows]
        new=not LOCAL_TELEMETRY.exists()
        with LOCAL_TELEMETRY.open('a',newline='') as f:
            w=csv.DictWriter(f,fieldnames=fields)
            if new: w.writeheader()
            w.writerow(dict(zip(fields,values))); f.flush(); os.fsync(f.fileno())
        if time.monotonic()-last_mirror>=30 or state_key!=last_state:
            mirror_telemetry(); last_mirror=time.monotonic(); last_state=state_key
        stop.wait(5)
    mirror_telemetry()
def run_monitored(cmd, local_log, checkpoint_file):
    with local_log.open('a') as logf: proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    stop = threading.Event(); thread = threading.Thread(target=monitor, args=(proc.pid, stop, checkpoint_file), daemon=True); thread.start()
    try:
        assert proc.stdout is not None
        with local_log.open('a') as logf:
            for line in proc.stdout:
                logf.write(line); logf.flush()
                if line.startswith('[') or line.startswith('  '): print(line, end='')
        rc = proc.wait()
    finally: stop.set(); thread.join(10)
    if rc: raise RuntimeError(f'runner return code {rc}; see {local_log}')
    return rc


In [ ]:
# Cell 4: restore model cache
TS_CACHE_ARCHIVE = GDRIVE/'cache'/'ts_weights_vertebrae_body_pp.tar.gz'; TS_HOME = Path.home()/'.totalsegmentator'
if TS_CACHE_ARCHIVE.is_file() and not TS_HOME.exists():
    with tarfile.open(TS_CACHE_ARCHIVE, 'r:gz') as tf: tf.extractall(Path.home())
print('Model cache ready or will be populated by smoke test.')


In [ ]:
# Cell 5: one-case smoke test
SMOKE_DIR = WORK_DIR/'smoke'; shutil.rmtree(SMOKE_DIR, ignore_errors=True); SMOKE_DIR.mkdir(parents=True)
smoke_manifest = SMOKE_DIR/'manifest.csv'
with MANIFEST.open() as src, smoke_manifest.open('w', newline='') as dst:
    reader=csv.DictReader(src); writer=csv.DictWriter(dst, fieldnames=reader.fieldnames); writer.writeheader(); writer.writerow(next(reader))
smoke_checkpoint=SMOKE_DIR/'results.csv'; t0=time.time()
run_monitored([sys.executable, '-u', str(RUNNER), '--manifest', str(smoke_manifest), '--out', str(SMOKE_DIR/'work'), '--checkpoint', str(smoke_checkpoint), '--events', str(LOG_DIR/f'smoke_events_{SESSION_ID}.jsonl'), '--active-state', str(ACTIVE_STATE), '--session-id', SESSION_ID, '--limit', '1'], SMOKE_DIR/'smoke.log', smoke_checkpoint)
smoke_rows=list(csv.DictReader(smoke_checkpoint.open())); assert len(smoke_rows)==1 and smoke_rows[0]['status']=='SUCCESS', smoke_rows
print(f'Smoke passed: {(time.time()-t0)/60:.1f} min')
if not TS_CACHE_ARCHIVE.exists() and TS_HOME.exists():
    TS_CACHE_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(TS_CACHE_ARCHIVE, 'w:gz') as tf: tf.add(TS_HOME, arcname='.totalsegmentator')


In [ ]:
# Cell 6: full resume by default; set RUN_SELECTIVE_QC=True in Cell 2 for mask-retaining QC reruns
RUN_LOG=WORK_DIR/'batch.log'; started=time.time(); completed=False
if RUN_SELECTIVE_QC:
    assert SELECTIVE_MANIFEST.is_file(), f'BLOCKING: missing {SELECTIVE_MANIFEST}'
    qc_root=TASK_DIR/'selective_qc_20260912'; qc_root.mkdir(parents=True,exist_ok=True)
    run_manifest=SELECTIVE_MANIFEST; run_checkpoint=qc_root/'results.csv'; run_out=WORK_DIR/'selective_cases'; run_events=qc_root/'case_task_events.jsonl'; archive=qc_root/'minimal_masks'
else:
    run_manifest=MANIFEST; run_checkpoint=CHECKPOINT_FILE; run_out=WORK_DIR/'cases'; run_events=EVENTS_FILE; archive=None
cmd=[sys.executable,'-u',str(RUNNER),'--manifest',str(run_manifest),'--out',str(run_out),'--checkpoint',str(run_checkpoint),'--events',str(run_events),'--active-state',str(ACTIVE_STATE),'--session-id',SESSION_ID]
if archive is not None: cmd += ['--minimal-archive',str(archive)]
try:
    run_monitored(cmd, RUN_LOG, run_checkpoint); completed=True
finally:
    elapsed_hours=(time.time()-started)/3600
    print(f'Batch cell ended: {elapsed_hours:.2f} h, estimated {elapsed_hours*CU_PER_HOUR:.2f} CU, checkpoint={run_checkpoint}')


In [ ]:
# Cell 7: result and resource summary
result_file=run_checkpoint if 'run_checkpoint' in globals() else CHECKPOINT_FILE
result_rows=list(csv.DictReader(result_file.open())); counts={}
for r in result_rows: counts[r['status']]=counts.get(r['status'],0)+1
tele=list(csv.DictReader(TELEMETRY_FILE.open())) if TELEMETRY_FILE.exists() else []
peak_rss=max((float(r['tree_rss_gib']) for r in tele if r.get('tree_rss_gib')),default=0); peak_pss=max((float(r['tree_pss_gib']) for r in tele if r.get('tree_pss_gib')),default=0); peak_uss=max((float(r['tree_uss_gib']) for r in tele if r.get('tree_uss_gib')),default=0); peak_gpu=max((float(r['gpu_mem_used_mib']) for r in tele if r.get('gpu_mem_used_mib') not in ('','None')),default=0); min_avail=min((float(r['system_available_gib']) for r in tele if r.get('system_available_gib')),default=0)
summary=f'''# Colab resource summary

- Notebook: {NOTEBOOK_VERSION}
- Session: {SESSION_ID}
- Mode: {'selective_qc' if RUN_SELECTIVE_QC else 'full_resume'}
- GPU: {gpu_name}; RAM: {ram_gb:.1f} GiB; VRAM: {vram_gb:.1f} GiB
- Observed rate: {CU_PER_HOUR} CU/h
- Results: {counts}
- Telemetry samples: {len(tele)}
- Peak runner process-tree RSS sum: {peak_rss:.2f} GiB
- Peak runner process-tree PSS sum: {peak_pss:.2f} GiB
- Peak runner process-tree USS sum: {peak_uss:.2f} GiB
- Minimum system available RAM: {min_avail:.2f} GiB
- Peak GPU memory: {peak_gpu:.0f} MiB
- Interpretation: PSS/USS are sampled estimates; Colab telemetry is not a cgroup kernel gold standard.
- Checkpoint: `{result_file}`
- Telemetry: `{TELEMETRY_FILE}`
- Events: `{run_events if 'run_events' in globals() else EVENTS_FILE}`
'''
summary_file=(qc_root/f'RESOURCE_SUMMARY_{SESSION_ID}.md') if RUN_SELECTIVE_QC else (LOG_DIR/f'RESOURCE_SUMMARY_{SESSION_ID}.md')
summary_file.write_text(summary); print(summary)
if completed:
    from google.colab import runtime
    print('Safe to disconnect: checkpoint and telemetry are on Drive.'); time.sleep(10); runtime.unassign()
